In [16]:
!pip install -q accelerate --upgrade
!pip install -q transformers --upgrade

In [17]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import datasets
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding, 
    pipeline
)

In [18]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BASE_DIR = "homeworks/HW13"
os.makedirs(f"{BASE_DIR}/artifacts", exist_ok=True)
print(f"reated/Verified: {BASE_DIR}/artifacts/")

Device: cuda
reated/Verified: homeworks/HW13/artifacts/


In [19]:
print(" Loading dataset...")
ds = load_dataset("dair-ai/emotion")
print(f"Dataset splits: {ds}")

label2id = {label: i for i, label in enumerate(ds["train"].features["label"].names)}
id2label = {v: k for k, v in label2id.items()}

print("Sanity Check:")
print(f"   Train size: {len(ds['train'])}")
print(f"   Validation size: {len(ds['validation'])}")
print(f"   Test size: {len(ds['test'])}")
print(f"   Classes ({len(label2id)}): {', '.join(list(id2label.values()))}")

df_head = ds["train"].to_pandas().head(5)
df_head["emotion"] = df_head["label"].map(id2label)
display(df_head[["text", "emotion"]])

print("Task description: Классификация коротких английских текстов (твитов) на 6 базовых эмоций.")

 Loading dataset...
Dataset splits: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
Sanity Check:
   Train size: 16000
   Validation size: 2000
   Test size: 2000
   Classes (6): sadness, joy, love, anger, fear, surprise


,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


Task description: Классификация коротких английских текстов (твитов) на 6 базовых эмоций.


In [20]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

demo_texts = [
    "I am feeling so happy today!",
    "This is absolutely terrible news, I can't believe it.",
    "Wait, what just happened? I didn't expect that.",
    "I'm really scared about the upcoming exam.",
    "My dog just brought me his favorite toy, so much love."
]

print("Tokenization Demo:\n" + "="*60)
for i, txt in enumerate(demo_texts[:3], 1):
    enc = tokenizer(txt)
    print(f"\nExample {i}: '{txt}'")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(enc['input_ids'])}")
    print(f"Input IDs: {enc['input_ids']}")
    print(f"Attention Mask: {enc['attention_mask']}")
    print(f"Special tokens: [CLS] (id {tokenizer.cls_token_id}), [SEP] (id {tokenizer.sep_token_id})")

print("--- Padding & Truncation Demo ---")
batch = tokenizer(demo_texts, padding="max_length", max_length=20, truncation=True, return_tensors="pt")
print(f"Original lengths: {[len(tokenizer(t)['input_ids']) for t in demo_texts]}")
print(f"Padded shape: {batch['input_ids'].shape}")
print(f"Padding token ID: {tokenizer.pad_token_id}")

Tokenization Demo:

Example 1: 'I am feeling so happy today!'
Tokens: ['[CLS]', 'i', 'am', 'feeling', 'so', 'happy', 'today', '!', '[SEP]']
Input IDs: [101, 1045, 2572, 3110, 2061, 3407, 2651, 999, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Special tokens: [CLS] (id 101), [SEP] (id 102)

Example 2: 'This is absolutely terrible news, I can't believe it.'
Tokens: ['[CLS]', 'this', 'is', 'absolutely', 'terrible', 'news', ',', 'i', 'can', "'", 't', 'believe', 'it', '.', '[SEP]']
Input IDs: [101, 2023, 2003, 7078, 6659, 2739, 1010, 1045, 2064, 1005, 1056, 2903, 2009, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Special tokens: [CLS] (id 101), [SEP] (id 102)

Example 3: 'Wait, what just happened? I didn't expect that.'
Tokens: ['[CLS]', 'wait', ',', 'what', 'just', 'happened', '?', 'i', 'didn', "'", 't', 'expect', 'that', '.', '[SEP]']
Input IDs: [101, 3524, 1010, 2054, 2074, 3047, 1029, 1045, 2134, 1005, 1056, 5987, 2008, 1012, 102]
Attention Mask: [1, 1, 1

In [21]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_ds = ds.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Dataset tokenized & collator ready.")

Dataset tokenized & collator ready.


In [22]:
print("Running inference with a pretrained BERT-like model:\n" + "="*60)
sentiment_pipe = pipeline(
    "text-classification", 
    model="distilbert-base-uncased-finetuned-sst-2-english", 
    device=0 if device.type=="cuda" else -1
)

for txt in demo_texts:
    res = sentiment_pipe(txt)[0]
    print(f"Text: '{txt}'\n  -> {res['label']} (conf: {res['score']:.3f})\n")

print("Вывод: Готовая модель предсказывает только POSITIVE/NEGATIVE. "
      "Она не различает тонкие эмоциональные категории (joy, sadness, fear и т.д.), "
      "поэтому для нашей задачи необходим fine-tuning на целевых метках.")

Running inference with a pretrained BERT-like model:


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: 'I am feeling so happy today!'
  -> POSITIVE (conf: 1.000)

Text: 'This is absolutely terrible news, I can't believe it.'
  -> NEGATIVE (conf: 0.997)

Text: 'Wait, what just happened? I didn't expect that.'
  -> NEGATIVE (conf: 0.992)

Text: 'I'm really scared about the upcoming exam.'
  -> NEGATIVE (conf: 0.999)

Text: 'My dog just brought me his favorite toy, so much love.'
  -> POSITIVE (conf: 1.000)

Вывод: Готовая модель предсказывает только POSITIVE/NEGATIVE. Она не различает тонкие эмоциональные категории (joy, sadness, fear и т.д.), поэтому для нашей задачи необходим fine-tuning на целевых метках.


In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir=f"{BASE_DIR}/outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",         
    seed=SEED,
    logging_steps=50,
    save_total_limit=2
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
train_result = trainer.train()
print("Training completed.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.476028,0.424836,0.926500,0.903703
2,0.243799,0.327024,0.936000,0.911260
3,0.182250,0.285628,0.936500,0.912214


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training completed.


In [24]:
print("Evaluating on TEST set...")

preds_output = trainer.predict(tokenized_ds["test"])

test_preds = np.argmax(preds_output.predictions, axis=-1)
test_true = preds_output.label_ids
test_probs = np.max(preds_output.predictions, axis=-1)

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average="macro")

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Macro:  {test_f1:.4f}")

Evaluating on TEST set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Accuracy: 0.9230
Test F1 Macro:  0.8787


In [25]:
print("Saving artifacts...")

# 1. Confusion Matrix
cm = confusion_matrix(test_true, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(id2label.values()))
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
cm_path = f"{BASE_DIR}/artifacts/confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.close()
print(f"Saved: {cm_path}")

# 2. Sample Predictions CSV
test_texts = ds["test"]["text"]
df_full_pred = pd.DataFrame({
    "text": test_texts,
    "true_label": [id2label[t] for t in test_true],
    "pred_label": [id2label[p] for p in test_preds],
    "confidence": test_probs
})

df_sample_pred = df_full_pred.sample(n=min(100, len(df_full_pred)), random_state=SEED)
csv_path = f"{BASE_DIR}/artifacts/sample_predictions.csv"
df_sample_pred.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

Saving artifacts...
Saved: homeworks/HW13/artifacts/confusion_matrix.png
Saved: homeworks/HW13/artifacts/sample_predictions.csv


In [26]:
errors_df = df_full_pred[df_full_pred["true_label"] != df_full_pred["pred_label"]]

print("5-10 примеров предсказаний модели:\n" + "="*60)
sample_preds = df_full_pred.sample(n=8, random_state=SEED)
for _, row in sample_preds.iterrows():
    status = "✅" if row["true_label"] == row["pred_label"] else "❌"
    print(f"{status} '{row['text']}'")
    print(f"   True: {row['true_label']} | Pred: {row['pred_label']} | Conf: {row['confidence']:.3f}\n")

print(f"Error Analysis: {len(errors_df)} ошибок / {len(df_full_pred)} всего")
print("Примеры ошибок:\n" + "="*60)

err_sample = errors_df.sample(n=min(5, len(errors_df)), random_state=SEED)
for _, row in err_sample.iterrows():
    print(f"❌ '{row['text']}'")
    print(f"   True: {row['true_label']} | Pred: {row['pred_label']} | Conf: {row['confidence']:.3f}\n")

print("Комментарий: Основные ошибки возникают на стыке семантически близких эмоций "
      "(joy↔love, fear↔surprise, sadness↔anger). Модель уверенно классифирует ярко "
      "выраженные маркеры, но затрудняется в контекстах с иронией, двойным отрицанием "
      "или смешанным аффектом. Для улучшения рекомендуется аугментация данных и "
      "увеличение контекстного окна.")

5-10 примеров предсказаний модели:
✅ 'i feel so dirty but after spending a day at the mk show me and a buddy decided we would get the two player starter between us luckily for us both i liked the everblight and he liked the circle maybe a tad to much so it all worked out well'
   True: sadness | Pred: sadness | Conf: 6.435

❌ 'i could feel his breath on me and smell the sweet scent of him'
   True: joy | Pred: love | Conf: 4.556

✅ 'i just want to feel loved by you'
   True: love | Pred: love | Conf: 5.329

❌ 'i have felt the need to write out my sometimes anxious feelings impatient thoughts lists of things that still should could be done before this baby arrives'
   True: anger | Pred: fear | Conf: 3.857

❌ 'at a party i met a girl who drew me to her'
   True: anger | Pred: fear | Conf: 1.699

❌ 'i feel this strange sort of liberation'
   True: surprise | Pred: fear | Conf: 3.535

✅ 'i remember feeling thrilled to use my nursing skills relieved that i could have a few days out of the 